### Question 1 :
Download the fashion-MNIST dataset and plot 1 sample image for each class as shown in the grid below. Use from keras.datasets import fashion_mnist for getting the fashion mnist dataset.

In [ ]:


# Find one image for each class
# unique_classes = np.unique(y_train)
# images_per_class = {}

# for cls in unique_classes:
#     # Find the first occurrence of each class
#     index = np.where(y_train == cls)[0][0]
#     images_per_class[cls] = x_train[index]

# Plot one image per class with labels
# plt.figure(figsize=(12, 8))
# for i, cls in enumerate(unique_classes):
#     plt.subplot(2, 5, i + 1)
#     plt.imshow(images_per_class[cls], cmap="gray")
#     plt.title(class_labels[cls])
#     plt.axis("off")

# plt.tight_layout()
# plt.show()
y_train

9

### Question 2
Implement a feedforward neural network which takes images from the fashion-mnist data as input and outputs a probability distribution over the 10 classes.
Your code should be flexible such that it is easy to change the number of hidden layers and the number of neurons in each hidden layer.

In [ ]:

import math
import random
import numpy as np
from keras.datasets import fashion_mnist
import numpy as np
import matplotlib.pyplot as plt

# Load the Fashion MNIST dataset
(x_train, y_train), (_, _) = fashion_mnist.load_data()

# Class labels for Fashion MNIST
class_labels = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]


class Variable:
    def __init__(self, data, _children=()):
        self.data = data
        self.children = set(_children)
        self.lable = ""
        self._backward = lambda: None
        self.grade = 0.0

    def __add__(self, other):
        if isinstance(other, (int, float)):
            other = Variable(other)
        out = Variable(self.data + other.data, (self, other))

        def backward():
            self.grade += out.grade
            other.grade += out.grade
        out._backward = backward
        return out

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            other = Variable(other)
        out = Variable(self.data * other.data, (self, other))

        def backward():
            self.grade += other.data*out.grade
            other.grade += self.data*out.grade
        out._backward = backward
        return out

    def __truediv__(self, other):
        return self * other**-1

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):  # other - self
        return other + (-self)

    def __rmul__(self, other):  # other * self
        return self * other

    def __radd__(self, other):  # other + self
        return self + other

    def __rtruediv__(self, other):  # other / self
        return other * self**-1

    def __pow__(self, other):
        if isinstance(other, (int, float)):
            out = Variable(self.data**other, (self,))

            def backward():
                self.grade += (other*self.data**(other-1))*out.grade

            out._backward = backward
            return out
        else:
            raise ValueError("Invalid value in pow opration")
            return null

    def exp(self):
        out = Variable(np.exp(self.data), (self,))

        def backward():
            self.grade += out.data*out.grade
        out._backward = backward
        return out
    
    def sigmoid(self):
        out = Variable(1/(1+np.exp(-self.data)), (self,))

        def backward():
            self.grade += out.data*(1-out.data)*out.grade

        out._backward = backward
        return out

    def __repr__(self):
        return f'Variable({self.data} and grade:{self.grade})'

    def backpropogation(self):
        self.grade = 1.0
        visited = []
        topological = []

        def dfs(node):
            visited.append(node)
            for child in node.children:
                if child not in visited:
                    dfs(child)
            topological.append(node)

        dfs(self)
        for node in reversed(topological):
            node._backward()



class Neuron:
    def __init__(self, input_size):
       
        self.weights = np.array([Variable(random.uniform(-0.1, 0.1))
                        for _ in range(input_size)])
        self.bias =  Variable(0) 

    def parameters(self):
        return np.concatenate([self.weights, np.array([self.bias])])     
    
    def forward(self, input):
        return np.dot(input, self.weights) + self.bias
    

class Layer:
    def __init__(self,num_neurons,num_weights,isoutput=False):
        self.isoutput = isoutput
        self.neurons = np.array([Neuron(num_weights) for _ in range(num_neurons)])

    def parameters(self):
       return np.concatenate([neuron.parameters() for neuron in self.neurons])

    def forward(self, inputs):
        outputs = []
        if self.isoutput == False:
         for i in inputs:
            outputs.append(np.array([neuron.forward(i).sigmoid() for neuron in self.neurons]))
        else:
         for i in inputs:
            temp = np.array([np.exp(neuron.forward(i)) for neuron in self.neurons])
            outputs.append(temp/np.sum(temp))
        
        return np.array(outputs)
    

class NeuralNet:
    def __init__(self, hidden):
        self.layers = []
        self.outputs = []
        for i in range(len(hidden)):
            if i == len(hidden)-1:
                self.layers.append(Layer(hidden[i][1],hidden[i][0],True))
            else:
                self.layers.append(Layer(hidden[i][1],hidden[i][0]))

    
    def parameters(self):
        return np.concatenate([layer.parameters() for layer in self.layers])

    def forward(self,inputs):
        for layer in self.layers:
            inputs = layer.forward(inputs)
        self.outputs = inputs
        return inputs

    def corss_loss(self, targets):
        self.loss = 0.0
        for target, output in zip(targets, self.outputs):
            self.loss += -np.log(output[target])  # Use the correct index for target class
        self.loss /= len(targets)  # Average over all samples
        return self.loss
    
    def MSE_loss(self,targets):
        self.loss = 0
        for target,output in zip(targets,self.outputs):
            self.loss += (target[0] - output[0])**2
        self.loss = self.loss/len(targets)
        return self.loss
    

    def accuracy(self,targets):
        predicted_labels = np.argmax(self.outputs, axis=1)

        # Count the number of correct predictions
        correct = np.sum(predicted_labels == targets)

        # Calculate accuracy as a percentage
        accuracy = (correct / len(targets)) * 100
        return accuracy
    
   
        
    def train(self, x_train, y_train, epochs, learning_rate , offset=1):
        for epoch in range(epochs):
            self.forward(x_train)
            loss = self.corss_loss(y_train)
            if epoch%offset == 0:
             print(f"Epoch {epoch+1}/{epochs} - Loss: {loss} - accuracy: {self.accuracy(y_train)}")
            loss.backpropogation()
            for parameter in self.parameters():
                parameter.data -= learning_rate*parameter.grade
                parameter.grade = 0.0
        




In [62]:
nn.forward([[10]])

array([[Variable(70.95795999872018 and grade:0.0)]], dtype=object)

In [57]:
nn.train(features, target, 100, 0.001 , 10)

Epoch 1/100 - Loss: Variable(66.6259043537415 and grade:0.0)
Epoch 11/100 - Loss: Variable(65.64778209816257 and grade:0.0)
Epoch 21/100 - Loss: Variable(64.69795108756233 and grade:0.0)
Epoch 31/100 - Loss: Variable(63.77541195618793 and grade:0.0)
Epoch 41/100 - Loss: Variable(62.879205794231694 and grade:0.0)
Epoch 51/100 - Loss: Variable(62.008412274966204 and grade:0.0)
Epoch 61/100 - Loss: Variable(61.16214788061774 and grade:0.0)
Epoch 71/100 - Loss: Variable(60.33956422109225 and grade:0.0)
Epoch 81/100 - Loss: Variable(59.53984644006215 and grade:0.0)
Epoch 91/100 - Loss: Variable(58.76221170328794 and grade:0.0)


In [76]:
y_train.shape

(60000,)

In [ ]:
x_train = x_train.reshape(x_train.shape[0], -1)
nn = NeuralNet([[784, 128],[128,128], [128, 128],[128,10]])
nn.train(x_train, y_train, 100, 0.01 , 10)


C:\Users\snehp\AppData\Local\Temp\ipykernel_12064\3561515007.py:79: RuntimeWarning: overflow encountered in exp
  out = Variable(1/(1+np.exp(-self.data)), (self,))
